# CS2311 — RL cho executor kiểu Agent0: GRPO + LoRA trên Kaggle, rồi benchmark lại MBPP+

Notebook này làm phần "executor học bằng RL" của Agent0 ở quy mô vừa một GPU T4.

1. **Dữ liệu = curriculum đã có.** Mỗi đề trong KB tự sinh được viết lại đúng dạng prompt MBPP+ của benchmark: mô tả + một assert ví dụ. Đề giống bài MBPP+ bị loại.
2. **Lọc đề ở biên năng lực**, như executor của Agent0: lấy 4 mẫu/đề từ chính model, giữ đề có p̂ ∈ [0.25, 0.75].
3. **GRPO + LoRA** (TRL): 8 rollout/đề.
   - Thưởng = tỉ lệ test qua, chạy trong **harness của project** (cách ly tiến trình, giới hạn CPU/RAM, chặn mạng), cộng 0,1 nếu có khối code nạp được (code bị cắt cụt không được thưởng).
   - Tuỳ chọn xấp xỉ ADPO: thưởng nhân s(p̂), bỏ chuẩn hoá theo độ lệch chuẩn.
4. **Benchmark lại MBPP+ (378 bài, EvalPlus)** cho model gốc và model sau RL, mỗi model 2 cấu hình:
   - `baseline`;
   - `retrieved` (2 ví dụ từ KB).

   Kết quả là bảng 2×2: tham số (RL) × ngữ cảnh (KB). So sánh cặp McNemar giữa model gốc và model sau RL.
5. Thêm pass@1 trên **tập val** (đề KB không dùng để huấn luyện) để đo trong phân phối.

```
Agent0 (paper)                              Notebook này
executor Qwen3-4B/8B, full fine-tuning      Qwen2.5-Coder-1.5B-Instruct + LoRA (r=16), fp16 trên T4
16 rollout × 128 đề/bước × 40 bước × 3 vòng  8 rollout × 4 đề/bước × ≤ 200 bước × 1 vòng
thưởng 1[đáp án = đáp án đa số]             thưởng = tỉ lệ test qua (test đã kiểm chứng) + 0.1·1[code nạp được]
lọc |p̂ − 0.5| ≤ 0.25 (k = 10)               lọc p̂ ∈ [0.25, 0.75] (k = 4), nới dần nếu quá ít đề
ADPO                                        tuỳ chọn ADPO_LIKE (co giãn advantage theo p̂) + clip-higher 0.28
rollout nhiều lượt có code interpreter       một lượt (đề code đã được chạy test khi chấm thưởng)
```

**Cách chạy:**
- **Accelerator: GPU T4 x2**, **Internet: On**.
- **Add Input**: dataset project (`CS2311_kaggle_upload.zip`).
- Nếu có hai GPU: GPU 0 lọc đề rồi huấn luyện; GPU 1 benchmark model gốc cùng lúc.
- Chạy thử trước: `LIMIT = 20`, `MAX_STEPS = 5`. Chạy thật: **Save & Run All (Commit)**.
- Chạy tiếp phiên trước: Add Input → output của version trước, đặt `RESUME_FROM = "auto"`.
- Tổng thời gian khoảng 4–6 giờ:
  - lọc đề khoảng 15–20 phút;
  - huấn luyện khoảng 1–1,5 phút/bước, chạy đến `TRAIN_HOURS`;
  - mỗi lần benchmark khoảng 10–20 phút;
  - chấm EvalPlus khoảng 3 phút mỗi lần.

In [ ]:
# ===== Cấu hình =====
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # model trên Hugging Face hoặc thư mục local (thử nhanh: ...-0.5B-Instruct)
LABEL = "qwen2.5-coder-1.5b"                   # kết quả: results/<LABEL>-hf (model gốc) và results/<LABEL>-grpo (sau RL)
KB_NAME = "all_models_clean"                   # KB làm dữ liệu huấn luyện và để truy hồi few-shot
EXTRA_KBS = []                                 # thêm KB (đường dẫn hoặc glob), vd KB của thí nghiệm Agent0 ở notebook chính:
                                               # ["/kaggle/input/**/kb_agent0/*/knowledge_base.jsonl"]
EXCLUDE_TEST_REPAIRED = True                   # bỏ mục KB có test từng bị sửa cho khớp lời giải
FRONTIER_FILTER = True                         # Agent0: chỉ học đề có p̂ của chính model trong [0.25, 0.75]
P_HAT_K = 4                                    # số mẫu/đề để đo p̂ (Agent0: 10)

MAX_STEPS = 200
TRAIN_HOURS = 5.0                              # ngân sách riêng cho huấn luyện (dừng ở cuối bước, lưu adapter)
LEARNING_RATE = 1e-5
LORA_R = 16
NUM_GENERATIONS = 8                            # rollout mỗi đề (Agent0: 16)
PROMPTS_PER_STEP = 4                           # số đề mỗi bước (Agent0: 128)
PER_DEVICE_BATCH = 4                           # completion mỗi micro-batch; hết bộ nhớ GPU thì 2 (grad_accum tự tăng)
MAX_COMPLETION = 512
KL_BETA = 0.0                                  # Agent0: 0.01 (chậm hơn ~30% vì tính thêm log-prob của model tham chiếu)
ADPO_LIKE = False                              # xấp xỉ ADPO: thưởng × s(p̂), không chuẩn hoá theo độ lệch chuẩn
REWARD = "fraction"                            # fraction: tỉ lệ test qua; binary: 1 nếu qua hết test
USE_VLLM = False                               # True: rollout bằng vLLM (cài thêm; trên T4 dễ lỗi, xem README)
RESUME_FROM = None                             # chạy tiếp phiên trước: "auto" (tìm trong /kaggle/input) hoặc đường dẫn
                                               # tới output cũ; nhớ tăng MAX_STEPS nếu phiên trước đã chạy đủ bước

EVAL_WITH_KB = True                            # thêm cấu hình retrieved cho cả hai model -> bảng 2×2 (cần Ollama cho embedding)
N_EXAMPLES = 2
LIMIT = None                                   # chạy thử: LIMIT = 20 (bài MBPP+), MAX_STEPS = 5
TIME_BUDGET_HOURS = 11.0                       # cả notebook (phiên Kaggle tối đa 12 giờ)
EVAL_RESERVE_HOURS = 1.5                       # giữ lại cho benchmark + chấm sau khi huấn luyện
ALLOW_CPU = False                              # chỉ để thử pipeline khi không có GPU (rất chậm)

RESULTS_DIR = "/kaggle/working/results"
WORK = "/kaggle/working/grpo"

In [ ]:
import os, sys, json, time, shutil, shlex, zipfile, subprocess, pathlib, urllib.request

DEADLINE = time.time() + TIME_BUDGET_HOURS * 3600

def hours_left():
    return max(0.0, (DEADLINE - time.time()) / 3600)

def run_live(cmd, cwd=None, env=None, check=False):
    """Chạy lệnh và in output ngay. Trả về mã thoát."""
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), cwd=cwd, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if "generation flags are not valid" in line or "FutureWarning" in line or "warnings.warn(" in line:
            continue
        print(line, end="", flush=True)
    code = proc.wait()
    if code != 0:
        print(f"⚠ lệnh kết thúc với mã {code}", flush=True)
        if check:
            raise RuntimeError(f"lệnh lỗi (mã {code}): {cmd[:4]}")
    return code

def start_job(name, cmds, gpu):
    """Chạy nền một chuỗi lệnh (lần lượt) trên một GPU; log ở WORK/logs/<name>.log."""
    log = pathlib.Path(WORK) / "logs" / f"{name}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    script = " ; ".join(shlex.join(list(map(str, c))) for c in cmds)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)) if gpu is not None else dict(os.environ)
    proc = subprocess.Popen(["bash", "-c", script], stdout=open(log, "w"), stderr=subprocess.STDOUT, env=env)
    print(f"[{name}] chạy nền trên {'GPU ' + str(gpu) if gpu is not None else 'CPU'} — log: {log}")
    return {"name": name, "proc": proc, "log": log}

def wait_job(job, tail=25):
    code = job["proc"].wait()
    lines = [l for l in job["log"].read_text(errors="replace").splitlines()
             if "generation flags are not valid" not in l and "Warning" not in l and "warnings.warn" not in l]
    print(f"[{job['name']}] xong (mã {code}). Cuối log:")
    print("\n".join(lines[-tail:]))
    return code

def find_project(root=pathlib.Path("/kaggle/input")):
    for p in root.rglob("agent0/__init__.py"):
        if p.parent.parent.name == "Source_Code":
            return p.parent.parent.parent
    for z in root.rglob("*.zip"):
        with zipfile.ZipFile(z) as zf:
            if any(n.endswith("Source_Code/agent0/__init__.py") for n in zf.namelist()):
                dest = pathlib.Path("/tmp/project_zip")
                shutil.rmtree(dest, ignore_errors=True)
                zf.extractall(dest)
                return next(dest.rglob("Source_Code/agent0/__init__.py")).parent.parent.parent
    raise FileNotFoundError("Không thấy project trong /kaggle/input. Add Input -> dataset chứa CS2311_kaggle_upload.zip")

def show_inputs(root=pathlib.Path("/kaggle/input"), depth=3, max_lines=60):
    """In các thư mục trong /kaggle/input (Kaggle gắn dataset ở /kaggle/input/datasets/<user>/<slug>/...)."""
    lines = [str(p) + "/" for p in sorted(root.rglob("*")) if p.is_dir() and len(p.relative_to(root).parts) <= depth
             and "Source_Code" not in p.parts]
    print("Input:", *(lines[:max_lines] or ["(trống)"]), sep="\n  ")

def previous_grpo(spec):
    """Thư mục grpo/ của output phiên trước (có run/checkpoints): "auto" = checkpoint có bước lớn nhất."""
    if not spec:
        return None
    if spec == "auto":
        best = None
        for ck in pathlib.Path("/kaggle/input").glob("**/grpo/run/checkpoints/checkpoint-*"):
            step = int(ck.name.rsplit("-", 1)[-1]) if ck.name.rsplit("-", 1)[-1].isdigit() else -1
            if best is None or step > best[0]:
                best = (step, ck.parents[2])
        if best is None:
            print("RESUME_FROM = 'auto': không thấy grpo/run/checkpoints trong /kaggle/input -> chạy mới")
            return None
        return best[1]
    p = pathlib.Path(spec)
    for cand in (p, p / "grpo", p.parent):
        if (cand / "run" / "checkpoints").exists():
            return cand
    raise FileNotFoundError(f"RESUME_FROM: không thấy run/checkpoints trong {p}")

show_inputs()
SRC = find_project()
PROJECT = pathlib.Path("/tmp/cs2311")
shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(SRC, PROJECT, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".env", "Report", "Slide"))
SOURCE = PROJECT / "Source_Code"
os.chdir(SOURCE)
sys.path.insert(0, str(SOURCE))
for d in (RESULTS_DIR, WORK):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
KB_PATH = PROJECT / "Data" / "knowledge_base" / KB_NAME / "knowledge_base.jsonl"
VARIANT = "_notr" if EXCLUDE_TEST_REPAIRED else ""
print("project:", SRC, "->", PROJECT, "| KB:", KB_PATH)

DATA, RUN = pathlib.Path(WORK) / "data", pathlib.Path(WORK) / "run"
BASE_LABEL, RL_LABEL = f"{LABEL}-hf", f"{LABEL}-grpo"
PREV = previous_grpo(RESUME_FROM)
RESUMED_DATA = False
if PREV is not None:
    # Chạy tiếp: lấy checkpoint, dữ liệu đã lọc (cùng tập train) và kết quả benchmark model gốc của phiên trước.
    # Không lấy kết quả của model sau RL: adapter sẽ khác sau khi học thêm, phải benchmark lại.
    if not (RUN / "checkpoints").exists():
        shutil.copytree(PREV / "run", RUN, dirs_exist_ok=True)
    if (PREV / "data" / "train.jsonl").exists():
        shutil.copytree(PREV / "data", DATA, dirs_exist_ok=True)
        RESUMED_DATA = True
    prev_results = PREV.parent / "results"
    if prev_results.exists():
        shutil.copytree(prev_results, RESULTS_DIR, dirs_exist_ok=True,
                        ignore=lambda d, names: [n for n in names if n in (RL_LABEL, "summary")])
    steps = sorted(int(c.name.rsplit("-", 1)[-1]) for c in (RUN / "checkpoints").glob("checkpoint-*"))
    print(f"Chạy tiếp từ {PREV}: checkpoint {steps[-1] if steps else '?'}, "
          f"{'dùng lại' if RESUMED_DATA else 'không có'} dữ liệu đã lọc")

In [ ]:
# Thư viện: TRL + PEFT (GRPO, LoRA) bản đã thử với code này, EvalPlus để chấm MBPP+.
# Ghim torch đang có sẵn: không để pip nâng torch (bản CUDA khác có thể không chạy với driver của Kaggle).
import importlib.metadata as _md
pkgs = ["trl==1.18.0", "peft==0.21.3", "evalplus==0.3.1", "python-dotenv", f"torch=={_md.version('torch')}"]
if USE_VLLM:
    pkgs.append("vllm")
run_live([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

import torch
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 2**30:.1f} GB, compute capability {p.major}.{p.minor}")
if N_GPU == 0 and not ALLOW_CPU:
    raise SystemExit("Không có GPU: bật Accelerator GPU T4 x2 (hoặc ALLOW_CPU = True chỉ để thử pipeline).")
TRAIN_GPU = 0 if N_GPU else None
EVAL_GPU = 1 if N_GPU >= 2 else TRAIN_GPU
PARALLEL = N_GPU >= 2
print("Huấn luyện trên", "GPU 0" if N_GPU else "CPU", "| benchmark model gốc",
      "song song trên GPU 1" if PARALLEL else "trước khi huấn luyện, cùng thiết bị")

In [ ]:
# Ollama chỉ để tính embedding (mxbai-embed-large) cho cấu hình retrieved — chạy trên CPU để GPU dành cho model.
if EVAL_WITH_KB:
    if not shutil.which("ollama"):
        run_live(["bash", "-c", "apt-get -qq update >/dev/null && apt-get -qq install -y zstd pciutils lshw >/dev/null; "
                                "curl -fsSL https://ollama.com/install.sh | sh"])
    ollama_env = dict(os.environ, CUDA_VISIBLE_DEVICES="", OLLAMA_NUM_PARALLEL="4", OLLAMA_KEEP_ALIVE="30m")
    ollama_env.pop("OLLAMA_API_KEY", None)
    OLLAMA = subprocess.Popen(["ollama", "serve"], stdout=open(pathlib.Path(WORK) / "ollama_server.log", "a"),
                              stderr=subprocess.STDOUT, env=ollama_env)
    for _ in range(90):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("Ollama không khởi động được — xem grpo/ollama_server.log")
    run_live(["ollama", "pull", "mxbai-embed-large"], check=True)
    # Dựng index embedding của KB một lần, trước khi nhiều tiến trình cùng dùng
    from agent0.scripts.run_benchmark import build_retriever
    r = build_retriever(KB_PATH, "mxbai-embed-large", "http://localhost:11434",
                        exclude_test_repaired=EXCLUDE_TEST_REPAIRED)
    print("Index KB:", r.stats)

In [ ]:
# Tải model (Hugging Face) vào /tmp — không tính vào output của notebook.
if pathlib.Path(MODEL_ID).exists():
    MODEL_PATH = str(MODEL_ID)
else:
    from huggingface_hub import snapshot_download
    MODEL_PATH = snapshot_download(MODEL_ID, local_dir=f"/tmp/hf_models/{MODEL_ID.replace('/', '__')}")
print("model:", MODEL_PATH)

import glob
extra = []
for k in EXTRA_KBS:
    found = sorted(glob.glob(str(k), recursive=True)) if any(c in str(k) for c in "*?[") else [str(k)]
    if not found or not all(pathlib.Path(f).exists() for f in found):
        raise FileNotFoundError(f"EXTRA_KBS: không thấy {k}")
    extra += found
KBS = [str(KB_PATH)] + extra
print("KB huấn luyện:", *KBS, sep="\n  ")
prep_common = [sys.executable, "tools/grpo_executor.py", "prepare", "--kb", *KBS, "--out_dir", DATA]
if not EXCLUDE_TEST_REPAIRED:
    prep_common.append("--keep_test_repaired")
# Chia train / val trước (nhanh, cố định theo hash) để benchmark model gốc dùng được val ngay
if RESUMED_DATA:
    print("Dùng lại train/val của phiên trước:", DATA)
else:
    _ = run_live(prep_common, check=True)

In [ ]:
# Benchmark model gốc: song song trên GPU 1 nếu có hai GPU, còn không thì chạy ngay bây giờ.
def bench_cmds(label, model, adapter=None):
    out_dir = pathlib.Path(RESULTS_DIR) / label
    common = [sys.executable, "tools/hf_generate.py", "--model", model, "--batch_size", 32, "--max_new_tokens", 1024,
              "--resume"]
    if adapter:
        common += ["--adapter", adapter]
    if LIMIT:
        common += ["--limit", LIMIT]
    cmds = [common + ["--output", out_dir / "baseline_samples.jsonl"]]
    if EVAL_WITH_KB:
        kb = common + ["--output", out_dir / f"with_kb_n{N_EXAMPLES}{VARIANT}_samples.jsonl",
                       "--example_selection", "retrieved", "--n_examples", N_EXAMPLES, "--kb_path", KB_PATH]
        if EXCLUDE_TEST_REPAIRED:
            kb.append("--exclude_test_repaired")
        cmds.append(kb)
    score = [sys.executable, "tools/grpo_executor.py", "score", "--model", model, "--tasks", DATA / "val.jsonl",
             "--out", pathlib.Path(WORK) / f"val_{label}.json", "--batch_size", 32]
    if adapter:
        score += ["--adapter", adapter]
    cmds.append(score)
    return cmds

base_job = None
if PARALLEL:
    base_job = start_job("bench_base", bench_cmds(BASE_LABEL, MODEL_PATH), EVAL_GPU)
else:
    for c in bench_cmds(BASE_LABEL, MODEL_PATH):
        run_live(c)

In [ ]:
# Lọc đề ở biên năng lực của chính model (p̂ từ P_HAT_K mẫu/đề, temperature 1.0 như rollout).
env_train = dict(os.environ, CUDA_VISIBLE_DEVICES=str(TRAIN_GPU)) if TRAIN_GPU is not None else dict(os.environ)
if FRONTIER_FILTER and not RESUMED_DATA:
    run_live(prep_common + ["--model", MODEL_PATH, "--p_hat_k", P_HAT_K, "--min_train", 32], env=env_train, check=True)
report = json.loads((DATA / "prepare_report.json").read_text())
print(json.dumps({k: report.get(k) for k in ("load", "n_train_pool", "n_val", "n_train", "frontier_rule")},
                 ensure_ascii=False, indent=1))

In [ ]:
# Huấn luyện GRPO + LoRA (GPU 0). Dừng khi đủ MAX_STEPS hoặc hết ngân sách; checkpoint mỗi 25 bước.
train_minutes = max(5.0, min(TRAIN_HOURS, hours_left() - EVAL_RESERVE_HOURS) * 60)
per_device = PER_DEVICE_BATCH
if (PROMPTS_PER_STEP * NUM_GENERATIONS) % per_device:
    raise ValueError("PROMPTS_PER_STEP * NUM_GENERATIONS phải chia hết cho PER_DEVICE_BATCH")
grad_accum = PROMPTS_PER_STEP * NUM_GENERATIONS // per_device
cmd = [sys.executable, "tools/grpo_executor.py", "train", "--model", MODEL_PATH, "--train", DATA / "train.jsonl",
       "--out_dir", RUN, "--max_steps", MAX_STEPS, "--time_budget_minutes", round(train_minutes, 1),
       "--lr", LEARNING_RATE, "--lora_r", LORA_R, "--lora_alpha", 2 * LORA_R, "--num_generations", NUM_GENERATIONS,
       "--per_device_batch", per_device, "--grad_accum", grad_accum, "--max_completion", MAX_COMPLETION,
       "--beta", KL_BETA, "--reward", REWARD, "--resume"]
if ADPO_LIKE:
    cmd.append("--adpo")
if USE_VLLM:
    cmd.append("--use_vllm")
print(f"Ngân sách huấn luyện: {train_minutes:.0f} phút")
train_code = run_live(cmd, env=env_train)

In [ ]:
# Benchmark model sau RL (adapter LoRA gộp vào model gốc khi nạp) trên GPU 0.
ADAPTER = RUN / "adapter"
if (ADAPTER / "adapter_config.json").exists():
    for c in bench_cmds(RL_LABEL, MODEL_PATH, adapter=str(ADAPTER)):
        run_live(c, env=env_train)
else:
    print("Không có adapter — huấn luyện chưa xong, bỏ qua benchmark model sau RL.")
if base_job is not None:
    wait_job(base_job)

In [ ]:
# Chấm EvalPlus (CPU) cho mọi lần chạy chưa chấm.
for samples in sorted(pathlib.Path(RESULTS_DIR).glob("*/*_samples.jsonl")):
    metrics = samples.with_name(samples.name.replace("_samples.jsonl", "_samples_metrics.json"))
    if metrics.exists() and metrics.stat().st_mtime > samples.stat().st_mtime:
        continue
    cmd = [sys.executable, "-m", "agent0.scripts.run_benchmark", "evaluate", "--samples_file", samples,
           "--evaluator", "local"]
    if LIMIT:
        cmd.append("--allow_incomplete")
    run_live(cmd)

In [ ]:
# Kết quả: bảng 2×2 (model gốc / sau RL × baseline / retrieved), McNemar giữa hai model, val trong phân phối.
from IPython.display import Markdown, Image, display
run_live([sys.executable, "tools/summarize_matrix.py", "--results_dir", RESULTS_DIR,
          "--model_pairs", f"{BASE_LABEL}:{RL_LABEL}"])
summary_md = pathlib.Path(RESULTS_DIR) / "summary" / "matrix_summary.md"
if summary_md.exists():
    display(Markdown(summary_md.read_text(encoding="utf-8")))
elif LIMIT:
    for m in sorted(pathlib.Path(RESULTS_DIR).glob("*/*_metrics.json")):
        d = json.loads(m.read_text())
        print(m.parent.name, m.name.replace("_samples_metrics.json", ""), "base", d["base_pass@1"],
              "plus", d["plus_pass@1"], "(PARTIAL)" if d.get("partial") else "")

vals = {}
for label in (BASE_LABEL, RL_LABEL):
    f = pathlib.Path(WORK) / f"val_{label}.json"
    if f.exists():
        vals[label] = json.loads(f.read_text())
if len(vals) == 2:
    a, b = vals[BASE_LABEL], vals[RL_LABEL]
    pa = {t["uid"]: t["passed"] for t in a["per_task"]}
    pb = {t["uid"]: t["passed"] for t in b["per_task"]}
    common = sorted(set(pa) & set(pb))
    gain = sum(1 for u in common if pb[u] and not pa[u])
    loss = sum(1 for u in common if pa[u] and not pb[u])
    from agent0.scripts.run_benchmark import mcnemar_exact
    print(f"Val ({len(common)} đề KB không dùng để huấn luyện): pass@1 {a['pass@1']} -> {b['pass@1']} | "
          f"tỉ lệ assert TB {a['mean_pass_frac']} -> {b['mean_pass_frac']} | +{gain} / -{loss} đề, "
          f"McNemar p = {mcnemar_exact(gain, loss):.3f}")

# Đường cong huấn luyện
log = RUN / "train_log.jsonl"
if log.exists():
    rows = [json.loads(l) for l in log.read_text().splitlines() if l.strip()]
    rows = [r for r in rows if "harness" in r]
    if rows:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        steps = [r["step"] for r in rows]
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.plot(steps, [r["harness"]["pass_frac"] for r in rows], label="tỉ lệ assert qua (rollout)")
        ax.plot(steps, [r["harness"]["all_pass"] for r in rows], label="qua hết test (rollout)")
        ax.set_xlabel("bước GRPO"); ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=0.3)
        fig.tight_layout()
        png = pathlib.Path(RESULTS_DIR) / "summary" / "grpo_training.png"
        png.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(png, dpi=130)
        display(Image(str(png)))

In [ ]:
# Đóng gói: kết quả benchmark + adapter LoRA + log huấn luyện + dữ liệu đã lọc (không kèm checkpoint, model).
if EVAL_WITH_KB:
    OLLAMA.terminate()
bundle = pathlib.Path("/kaggle/working/cs2311_grpo_bundle")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(RESULTS_DIR, bundle / "results")
for sub in ("data", "logs"):
    if (pathlib.Path(WORK) / sub).exists():
        shutil.copytree(pathlib.Path(WORK) / sub, bundle / "grpo" / sub)
for f in list(pathlib.Path(WORK).glob("val_*.json")) + [RUN / "train_log.jsonl", RUN / "train_summary.json"]:
    if f.exists():
        (bundle / "grpo").mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, bundle / "grpo" / f.name)
if (RUN / "adapter").exists():
    shutil.copytree(RUN / "adapter", bundle / "grpo" / "adapter")
archive = shutil.make_archive("/kaggle/working/cs2311_grpo_results", "zip", bundle)
shutil.rmtree(bundle, ignore_errors=True)
print(archive, round(os.path.getsize(archive) / 1e6, 1), "MB")
n_files = sum(1 for p in pathlib.Path("/kaggle/working").rglob("*") if p.is_file())
size_gb = sum(p.stat().st_size for p in pathlib.Path("/kaggle/working").rglob("*") if p.is_file()) / 1e9
print(f"/kaggle/working: {n_files} file, {size_gb:.2f} GB (Kaggle lưu tối đa 20 GB)")